In [53]:
import pandas as pd
import numpy as np

In [54]:
# FEATURE 1: The Transaction Parser

def parse_transactions(filepath):
    df = pd.read_csv(filepath)

    df = df.drop_duplicates()

    df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

    df['hour'] = df['Time'].astype(str).str[:2].astype(int)
    df['month'] = df['date'].dt.month

    df['Amount'] = df['Amount'].astype(str) \
                               .str.replace('Rs.', '', regex=False) \
                               .str.replace(',', '', regex=False) \
                               .str.replace('$', '', regex=False) \
                               .str.strip()
    df['amount'] = pd.to_numeric(df['Amount'], errors='coerce')

    df['Type'] = df['Type'].astype(str).str.lower().str.strip()
    df['type'] = df['Type'].map({'dr': 'debit', 'debit': 'debit', 'cr': 'credit', 'credit': 'credit'})

    return df

In [55]:
# FEATURE 2: Vendor Extractor

vendor_dict = {
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Zepto': ['ZEPTO'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'Myntra': ['MYNTRA'],
    'Uber': ['UBER'],
    'Ola': ['OLA'],
    'Zerodha': ['ZERODHA'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Jio': ['JIO'],
    'Airtel': ['AIRTEL'],
    'Starbucks': ['STARBUCKS'],
    'ThirdWave': ['THIRD WAVE', 'THIRDWAVE']
}

def extract_vendor(desc):
    desc_upper = str(desc).upper()

    if 'ATM-WDL' in desc_upper or 'ATM WDL' in desc_upper:
        return 'Cash Withdrawal'

    for vendor, keywords in vendor_dict.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor

    if 'UPI-' in desc_upper and '@' in desc_upper:
        return 'P2P Transfer'

    return 'Uncategorised'

In [56]:
# FEATURE 3: Category Tagger

category_map = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Myntra': 'E-commerce',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Starbucks': 'Cafe',
    'ThirdWave': 'Cafe',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Jio': 'Utilities',
    'Airtel': 'Utilities',
    'Zerodha': 'Investments',
    'BookMyShow': 'Entertainment',
    'Cash Withdrawal': 'Cash Withdrawal',
    'P2P Transfer': 'Personal Transfer'
}

In [57]:
# ==========================================
# MAIN EXECUTION & FEATURE 4-8 LOGIC
# ==========================================
def run_spend_dna():
    # 1. Parse Data
    df = parse_transactions("Data set for DADS June.csv")

    # 2. Extract Vendors
    df['vendor_clean'] = df['Description'].apply(extract_vendor)

    # 3. Tag Categories
    df['category'] = df['vendor_clean'].map(category_map).fillna('Uncategorised')

    # Separate debits for spending analysis
    debits = df[df['type'] == 'debit'].copy()

    # 4. Spending Overview
    total_credits = df[df['type'] == 'credit']['amount'].sum()
    total_debits = debits['amount'].sum()
    net_change = total_credits - total_debits
    savings_rate = (net_change / total_credits) * 100 if total_credits > 0 else 0

    # Aggregate categories (excluding P2P and Cash Withdrawals for pure consumption)
    consumption = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Uncategorised'])]
    cat_totals = consumption.groupby('category')['amount'].sum().sort_values(ascending=False)
    vendor_totals = consumption.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

    # 5. Monthly Trend Analysis (Pivot)
    month_pivot = consumption.pivot_table(values='amount', index='category', columns='month', aggfunc='sum', fill_value=0)

    # 6. Time-of-Day Patterns
    # Calculate late night food delivery percentage
    food_delivery = debits[debits['category'] == 'Food Delivery']
    late_night_food = food_delivery[(food_delivery['hour'] >= 21) | (food_delivery['hour'] <= 2)]
    late_night_food_pct = (len(late_night_food) / len(food_delivery)) * 100 if len(food_delivery) > 0 else 0

    # 7. Anomaly Detection (Z-Score)
    debits['cat_mean'] = debits.groupby('category')['amount'].transform('mean')
    debits['cat_std'] = debits.groupby('category')['amount'].transform('std')
    debits['z_score'] = (debits['amount'] - debits['cat_mean']) / debits['cat_std']
    anomalies = debits[debits['z_score'] > 2].sort_values(by='z_score', ascending=False).head(5)

    # 8. Spending Archetypes
    archetypes = []

    food_total = cat_totals.get('Food Delivery', 0) + cat_totals.get('Restaurants', 0) + cat_totals.get('Cafe', 0)
    if (food_total / total_debits) > 0.25:
        archetypes.append(("THE FOODIE", f"({(food_total/total_debits)*100:.1f}% on food)"))

    if (cat_totals.get('Quick Commerce', 0) / total_debits) > 0.15:
        archetypes.append(("THE QUICK COMMERCE JUNKIE", f"({(cat_totals.get('Quick Commerce', 0)/total_debits)*100:.1f}% on Q-com)"))

    if (cat_totals.get('E-commerce', 0) / total_debits) > 0.15:
        archetypes.append(("THE SHOPAHOLIC", f"({(cat_totals.get('E-commerce', 0)/total_debits)*100:.1f}% on e-commerce)"))

    if (cat_totals.get('Investments', 0) / total_debits) > 0.15:
        archetypes.append(("THE INVESTOR", f"({(cat_totals.get('Investments', 0)/total_debits)*100:.1f}% on SIPs/Stocks)"))

    if late_night_food_pct > 50:
        archetypes.append(("THE LATE-NIGHT SNACKER", f"({late_night_food_pct:.0f}% food after 9 PM)"))

    if savings_rate < 10:
        archetypes.append(("THE YOLO SPENDER", f"(savings rate {savings_rate:.1f}%)"))

    # Bonus Archetype: The Pavement Coffee Connoisseur (Bengaluru specific)
    if vendor_totals.get('ThirdWave', {}).get('count', 0) > 10 or vendor_totals.get('Starbucks', {}).get('count', 0) > 10:
        archetypes.append(("THE PAVEMENT COFFEE CONNOISSEUR", "(High frequency of premium cafe visits)"))

    # ==========================================
    # FINAL PRINTED REPORT
    # ==========================================
    print("=" * 66)
    print(f"{'SpendDNA REPORT':^66}")
    print(f"{'Jan to Jun 2024':^66}")
    print(f"{f'6 months | {len(df)} transactions':^66}")
    print("=" * 66)

    print("\nEXECUTIVE SUMMARY")
    print("-" * 20)
    print(f"Total credits  : Rs. {total_credits:,.0f}")
    print(f"Total debits   : Rs. {total_debits:,.0f}")

    if net_change < 0:
        print(f"Net change     : Rs. {abs(net_change):,.0f} (overspending)")
    else:
        print(f"Net change     : Rs. {net_change:,.0f} (saved)")

    print(f"Savings rate   : {savings_rate:.1f}%")
    print(f"Transactions   : {len(df)}")
    print(f"Unique vendors : {df['vendor_clean'].nunique()}")

    print("\nTOP CATEGORIES (% of debit total)")
    print("-" * 35)
    for cat, amt in cat_totals.head(5).items():
        pct = (amt / total_debits) * 100
        bars = "#" * int(pct / 1.5)
        print(f"{cat:<15} {bars:<15} {pct:>5.1f}%   Rs. {amt:,.0f}")

    print("\nTOP VENDORS")
    print("-" * 20)
    for vendor, row in vendor_totals.head(5).iterrows():
        print(f"{vendor:<15} Rs. {row['sum']:>8,.0f}   ({row['count']:.0f} orders)")

    print("\nTIME-OF-DAY PATTERNS")
    print("-" * 25)
    print(f"Food Delivery peaks : 21:00 - 01:00 ({late_night_food_pct:.0f}% of orders)")

    print("\nTOP ANOMALIES (2+ stddev from category mean)")
    print("-" * 45)

    # Format the dates directly in the dataframe before looping
    anomalies['date_str'] = anomalies['date'].dt.strftime('%d %b')

    for _, row in anomalies.iterrows():
        print(f"{row['date_str']:<8} {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f} (z={row['z_score']:.1f})")
    print("\nSPENDING ARCHETYPES")
    print("-" * 25)
    for arch, metric in archetypes:
        print(f"-> {arch:<25} {metric}")

    print("\n" + "=" * 66)

# Run the analyzer
run_spend_dna()

                         SpendDNA REPORT                          
                         Jan to Jun 2024                          
                   6 months | 1310 transactions                   

EXECUTIVE SUMMARY
--------------------
Total credits  : Rs. 254,818
Total debits   : Rs. 1,236,193
Net change     : Rs. 981,375 (overspending)
Savings rate   : -385.1%
Transactions   : 1310
Unique vendors : 19

TOP CATEGORIES (% of debit total)
-----------------------------------
E-commerce      ###############  23.6%   Rs. 292,190
Investments     #########        14.6%   Rs. 180,000
Food Delivery   #####             8.3%   Rs. 102,135
Quick Commerce  ##                3.6%   Rs. 44,637
Transport       #                 2.6%   Rs. 32,709

TOP VENDORS
--------------------
Amazon          Rs.  240,824   (62 orders)
Zerodha         Rs.  180,000   (12 orders)
Swiggy          Rs.   65,448   (153 orders)
Myntra          Rs.   51,366   (12 orders)
Zomato          Rs.   36,687   (82 orders)

TIM